> Техническая проверка 14.09.2026: этот файл содержит только текст и ссылки; расчётных ячеек нет. HTML пересобран, новый расчёт этим не выполнен.

# 20.11 Прямая КТ/FEM-чувствительность боковых сборок

> **Статус:** контракт отдельной прямой задачи; полноценный параметрический
> расчёт не реализован. В MATLAB-ветке имеется только локальный Якобиан внутри
> обратной локализации `20.10`. Он полезен как предварительная диагностика, но
> не заменяет исследование влияния параметров анатомической модели на сигналы
> боковых сборок.

До исправления атрибуции 90/100 мм, подтверждения приборных параметров и
воспроизводимого запуска `20.10` численные выводы этой ветки считать
исследовательскими гипотезами.


## Назначение и границы задачи

Для фиксированной индивидуальной КТ/FEM-геометрии требуется вычислить, как
изменяются абсолютный импеданс и его физиологические приращения при явном
изменении одного или нескольких параметров модели:

$$
Z_L = F_L(\boldsymbol\theta,\,G,\,P,\,D,\,S),
$$

где $L$ — размер сборки, $\boldsymbol\theta$ — электрические свойства тканей,
$G$ — анатомическая геометрия, $P$ — поза матриц, $D$ — параметры прибора и
электродного контакта, $S$ — физиологическое состояние.

Границы строго разделены:

- `20.10` восстанавливает позу и эффективные параметры по данным — это
  обратная задача;
- `20.11` меняет заранее заданные параметры и вычисляет ответ — это прямая
  задача чувствительности;
- `20.12` определяет критерий нарушения плоскослоистого приближения и
  $L_{max}$;
- серия `32` выбирает информативные размеры; чувствительность является одним
  из её входов, но не готовым решением выбора.

Первый расчёт относится к эксперименту 2 и реокардиомонитору МГТУ. Для
эксперимента 3 требуется отдельный запуск с реокардиомонитором РНЦХ; одинаковая
схема электродов не доказывает одинаковость масштаба, частоты и контактов.


## Что уже есть в коде и чего нет

Текущий notebook содержит только Markdown и не выполняет вычислений.

В
[`run_trkg4_inverse_inhale.m`](../MATLAB_TRKG4_real_subjects/src/run_trkg4_inverse_inhale.m)
после подгонки вычисляется локальный конечно-разностный Якобиан

$$
J_{Lj}=\frac{\partial Z_L}{\partial x_j},\qquad
\mathbf{x}=(\rho_{soft},\rho_{lung},u,v,\varphi).
$$

Код сохраняет производные по двум эффективным сопротивлениям и трём
параметрам позы, сингулярные числа и локальную оценку обусловленности. Это
касательная характеристика около одного найденного кандидата, а не анализ
реальной вариабельности анатомии или всех тканей.

Ограничения этого фрагмента:

1. шаги конечных разностей и масштабы параметров заданы вручную; сходимость по
   шагу не проверяется;
2. для CRLB принято `assumed_sigma_ohm = 5`. Это не измеренный шум прибора и не
   оценка разброса записей, а неподтверждённое допущение о независимой
   одинаковой ошибке каждой точки $Z(L)$; до экспериментального обоснования
   такую CRLB нельзя трактовать как погрешность параметров;
3. меняются только $\rho_{soft}$, $\rho_{lung}$ и поза; свойства костей,
   сердца, крови, контакт, частота и границы масок фиксированы;
4. расчёт наследует ошибочную атрибуцию 90/100 мм и исторический локальный
   кандидат `20.10`;
5. [`run_trkg4_forward_subject.m`](../MATLAB_TRKG4_real_subjects/src/run_trkg4_forward_subject.m)
   выполняет один прямой расчёт, но не организует параметрический sweep.

Следовательно, существующие файлы `*_jacobian_*` и
`*_parameter_sensitivity_*` сохраняются как промежуточные outputs inverse-задачи,
а не как законченный результат `20.11`.


## Проверяемые вопросы

Расчёт должен отвечать отдельно на следующие вопросы:

1. Как каждый параметр меняет абсолютный $Z(L)$ для каждого реально
   использованного размера?
2. Как параметры меняют дыхательное приращение
   $\Delta Z_{resp}=Z_{inhale}-Z_{exhale}$?
3. Как перенос заданных пульсовых $\Delta\rho_{soft}$ и
   $\Delta\rho_{lung}$ меняет $\Delta Z_{pulse}$?
4. Какие размеры по-разному чувствительны к мягким тканям и лёгкому, а какие
   дают почти коллинеарные отклики?
5. Насколько результат зависит от рёбер, конечной границы лёгкого, формы тела,
   контакта и позы?
6. Где линейный перенос записанного приращения через Якобиан перестаёт
   совпадать с полным нелинейным FEM-расчётом?

«Подписанная тканевая чувствительность» означает производную по явно названной
величине — $\partial Z/\partial\sigma_k$ или
$\partial Z/\partial\rho_k$. Знак зависит от параметризации и монтажа. Его
нельзя автоматически называть долей сигнала конкретного органа или определять
по одной лишь синхронности с ЭКГ.


## Входной контракт

Для каждого запуска обязательны:

- деидентифицированный `subject_id`, эксперимент, прибор, канал, состояние
  дыхания и `run_id`;
- принятые результаты `20.01`, `20.02` и версия кандидата позы `20.10` с
  полным манифестом и хешами;
- внешние маски тела, лёгких, костей, сердца и при наличии крови, FEM-сетка и
  отчёт геометрического QC;
- фактические размеры и геометрия электродов по субъектному QC: для
  добровольца с дубликатом сохраняется 90 мм и исключается поздняя копия
  100 мм; у второго добровольца обе записи самостоятельны;
- подтверждённая частота прибора, ток, знак/единицы канала, площадь электродов
  и модель контактного импеданса;
- исходные и диапазонные значения электрических свойств с частотой,
  температурой, литературным или экспериментальным источником и статусом;
- принятые дыхательные и ЭКГ-разметки серии `11`, если моделируются реальные
  приращения;
- версия MATLAB, EIDORS, mesher, кода и параметры сетки.

В текущем `trkg4_config.m` частота 50 кГц, контактные параметры и часть
тканевых свойств заданы жёстко. Пока частота реокардиомонитора МГТУ не
подтверждена документацией, 50 кГц является модельным допущением, а не фактом
эксперимента. Значения `rho_soft` и `rho_lung` происходят из прежних solution
clouds/COMSOL-заметок; использовать их одновременно как неподвижную истину и
как подтверждение той же модели нельзя.


## Группы параметров

Параметры менять раздельными именованными сценариями, а не одним общим
«шумом».

| Группа | Минимальный состав | Что фиксировать |
|---|---|---|
| Электрические свойства | $\sigma$ или $\rho$ мягких тканей, лёгкого, кости, сердца и крови | частота, температура, комплексная или скалярная модель, источник диапазона |
| Электрод и прибор | контактный импеданс, площадь, ток, gain, знак и масштаб | измерено, взято из документации или принято как допущение |
| Поза | $u$, $v$, $\varphi$, ошибка центра, ориентации и расстояний | ручная наклейка и переклейка не заменяются случайным числом без данных |
| Геометрия | границы масок, $h(s)$, рёбра, конечный объём лёгкого, качество сетки | способ морфологического возмущения и физически допустимый диапазон |
| Состояние | вдох/выдох, последовательность записей, пульсовые $\Delta\rho$ | что измерено в этой сессии, а что перенесено из другой записи |

Если маски крови нет, соответствующий расчёт следует назвать сценарием
«кровь включена в фон мягких тканей», а не моделью с независимо заданной
кровью. Одной КТ на вдохе недостаточно, чтобы получить реальную дыхательную
или пульсовую динамику; недостающие состояния задаются только как сценарии до
появления независимой проверки.


## Минимальный математический протокол

Для скалярного параметра $p$ сначала вычислять центральную разность

$$
S_p(L)=\frac{F_L(p+\delta p)-F_L(p-\delta p)}{2\delta p}
$$

и безразмерную чувствительность

$$
S_p^{norm}(L)=\frac{p}{Z_L}S_p(L).
$$

Обязателен тест как минимум на двух меньших шагах $\delta p$. Производную не
принимать, если знак или порядок величины нестабилен относительно шага либо
качества сетки.

Для измеренного или заданного в сценарии вектора приращений допускается
линейная предварительная оценка

$$
\Delta\mathbf Z_{lin}=J\,\Delta\boldsymbol\theta.
$$

Её нужно сравнить с полным решением
$\Delta\mathbf Z_{FEM}=F(\boldsymbol\theta+\Delta\boldsymbol\theta)-
F(\boldsymbol\theta)$ и сообщить ошибку линеаризации по каждому $L$.

До появления обоснованных вероятностных распределений анализ вести по
диапазонам и детерминированным сценариям. Вариабельность физиологии, дрейф,
контакт, ошибка наклейки, ошибка сегментации и численная ошибка FEM — разные
источники неопределённости; их нельзя складывать в одну гауссовскую величину
без данных и модели.


## Последовательность расчёта

1. Исправить и воспроизвести `20.10`; сохранить позу как фиксированный вход
   первого sensitivity-run.
2. На синтетической геометрии проверить знак, единицы и сходимость конечных
   разностей.
3. Выполнить однопараметрические локальные производные на индивидуальной
   КТ-модели.
4. Пройти физически обоснованные интервалы ключевых параметров полным FEM и
   проверить нелинейность.
5. Только для выявленных связанных параметров добавить двумерные профили или
   выбранные взаимодействия; полный комбинаторный перебор заранее не нужен.
6. Отдельно выполнить статический, дыхательный и пульсовый сценарии, не
   смешивая их входы.
7. Повторить анализ на независимой анатомии. Эксперимент 3 считать отдельной
   приборной веткой, даже если геометрическая схема электродов совпадает.


## Выходы и хранение

Численные результаты хранить вне Git, например в
`derived_root/ct_fem/sensitivity/<experiment>/<subject_id>/<run_id>/`.

Минимальный комплект:

- `manifest.json` с хешами входов, версиями и статусом каждого допущения;
- таблица базовых $Z(L)$ и всех возмущённых расчётов;
- размерный и нормированный Якобиан с точным определением параметризации;
- таблица сходимости по шагу и сетке;
- сравнение линейного переноса с полным FEM;
- отдельные результаты статического, дыхательного и пульсового сценариев;
- реестр исключённых запусков и причин;
- краткий деидентифицированный отчёт с границами применимости.

В репозитории оставлять только код, схему конфигурации и обезличенный
манифест. STL, первичные сигналы, координаты, сохранённые FEM-поля и любые
субъектно-связанные outputs не коммитить.


## Критерии принятия

Этап не считается выполненным, пока не показано одновременно:

1. все входы и единицы прослеживаются до прибора, КТ/сегментации или явно
   названного допущения;
2. исправлена запись 90/100 мм и результат не зависит от устаревших outputs;
3. подтверждены частота, контактная модель и тканевые свойства либо показана
   чувствительность к их допустимым диапазонам;
4. конечные разности сходятся по шагу, а FEM — по сетке;
5. поза фиксирована или её неопределённость варьируется отдельно, без повторной
   скрытой подгонки под тот же сигнал;
6. линейный перенос проверен полным нелинейным расчётом;
7. результат повторён хотя бы на второй индивидуальной анатомии до любых
   заявлений об общем диапазоне размеров;
8. численная чувствительность не названа экспериментальной валидацией или
   доказанной долей сигнала органа.

После выполнения этого контракта outputs поступают в `20.12`, серии `31–32` и
план проверки `34`. До этого `20.11` остаётся постановкой метода.


## Происхождение

- **Основной эксперимент:** 2, МГТУ, реокардиомонитор МГТУ.
- **Отдельная проверка:** эксперимент 3, РНЦХ, реокардиомонитор РНЦХ.
- **Каноническая вычислительная ветка:** `MATLAB_TRKG4_real_subjects/`.
- **Исторический исходный коммит миграции:**
  `1ac7813654b59861e359cffe3ad40941baeab2d9`.
- **Статус утверждений:** наблюдаемая реализация локального Якобиана — факт о
  коде; пригодность выбранных шагов, 5 Ом, 50 кГц и параметров контакта —
  непроверенные допущения; полноценная прямая чувствительность — предстоящая
  работа.
